# Stage 4 — Rejection-sampling SFT

Use the RL'd model from notebook 04 as a **data factory**:

1. Generate many completions per prompt.
2. Keep only the ones that pass the reward checker.
3. Run a fresh round of supervised fine-tuning on the filtered set.

No new reward signal — just *cleaner data*. R1 used this to stabilize the model before the final RL pass.

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F, random, re
torch.manual_seed(0); random.seed(0)

VOCAB = list('0123456789+=TA .')
stoi = {c: i for i, c in enumerate(VOCAB)}; itos = {i: c for c, i in stoi.items()}
V, PAD, EOS, BLOCK = len(VOCAB), stoi[' '], stoi['.'], 24
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)

class TinyLM(nn.Module):
    def __init__(self, V, d=64, h=4, L=2, block=BLOCK):
        super().__init__()
        self.tok = nn.Embedding(V, d); self.pos = nn.Embedding(block, d)
        self.blocks = nn.ModuleList([nn.TransformerEncoderLayer(d, h, dim_feedforward=4*d, batch_first=True, activation='gelu') for _ in range(L)])
        self.head = nn.Linear(d, V)
    def forward(self, x):
        T = x.shape[1]
        h = self.tok(x) + self.pos(torch.arange(T, device=x.device))
        mask = nn.Transformer.generate_square_subsequent_mask(T).to(x.device)
        for blk in self.blocks: h = blk(h, src_mask=mask)
        return self.head(h)

model = TinyLM(V)
model.load_state_dict(torch.load('after_grpo.pt'))

<All keys matched successfully>

In [2]:
# --- Step 1: Generate many completions, keep only correct ones ---
@torch.no_grad()
def sample(prompt, max_new=16, temperature=1.0):
    x = torch.tensor([encode(prompt)])
    for _ in range(max_new):
        logits = model(x[:, -BLOCK:])[:, -1, :] / temperature
        nxt = torch.multinomial(F.softmax(logits, -1), 1)
        x = torch.cat([x, nxt], 1)
        if nxt.item() == EOS: break
    return decode(x[0].tolist())

def is_correct(text, a, b):
    m = re.search(r'A(\d+)\.', text)
    return m is not None and int(m.group(1)) == a + b

kept = []
for a in range(10):
    for b in range(10):
        for _ in range(8):                       # 8 attempts per prompt
            out = sample(f'{a}+{b}=')
            if is_correct(out, a, b):
                kept.append(out)
                break
print(f'kept {len(kept)} / 100 prompts')
for ex in kept[:5]: print(' ', ex)

kept 27 / 100 prompts
  0+1=T0+1=A1.
  0+2=T0+21A2.
  0+4=T0+18A4.
  0+6=T0+68A6.
  0+7=T0+73A7.


In [3]:
# --- Step 2: SFT on the filtered set ---
def pad_batch(strings):
    ids = [encode(s) for s in strings]
    L = max(len(x) for x in ids)
    return torch.tensor([x + [PAD]*(L-len(x)) for x in ids])

opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
for step in range(300):
    batch = random.choices(kept, k=16)
    x = pad_batch(batch)
    logits = model(x[:, :-1])
    loss = F.cross_entropy(logits.reshape(-1, V), x[:, 1:].reshape(-1), ignore_index=PAD)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 60 == 0: print(f'step {step:3d}  loss {loss.item():.3f}')

step   0  loss 0.638


step  60  loss 0.131


step 120  loss 0.103


step 180  loss 0.098


step 240  loss 0.099


In [4]:
@torch.no_grad()
def greedy(prompt, max_new=16):
    x = torch.tensor([encode(prompt)])
    for _ in range(max_new):
        logits = model(x[:, -BLOCK:])[:, -1, :]
        nxt = torch.argmax(logits, -1, keepdim=True)
        x = torch.cat([x, nxt], 1)
        if nxt.item() == EOS: break
    return decode(x[0].tolist())

def pass_rate(n=100):
    c = 0
    for _ in range(n):
        a, b = random.randint(0,9), random.randint(0,9)
        if is_correct(greedy(f'{a}+{b}='), a, b): c += 1
    return c / n

print(f'pass-rate after rejection-sampling SFT: {pass_rate():.0%}')
torch.save(model.state_dict(), 'after_reject_sft.pt')

pass-rate after rejection-sampling SFT: 26%


## Takeaway

The model just trained on its **own correct outputs**. No new labels. No new reward.

This is how R1 turns an unstable RL'd model into a clean, deployable one — and it's the same trick used by every modern self-improvement loop.